# snapshot

> Flat namespace serializer

The read pipeline: a namespace dict in, `VarInfo` rows out — always one lazy level at a
time. `snapshot` lists top-level bindings; `expand` walks a positional accessor and pages
its children (`MAX_CHILDREN` per call, trailing load-more sentinel); `grid_page` windows a
gridable object. `profile_view` groups rows PyCharm-style (data / Modules / Types /
Functions / Special Variables) with per-profile visibility (`minimal`/`standard`/`full`).
Nothing here mutates or holds references — every call re-reads the live namespace.

In [ ]:
#| default_exp snapshot

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
from paar.snapshot import snapshot, expand, grid_page, category, profile_view
import math, pathlib, numpy as np, pandas as pd
def test_sorted_and_fields():
    rows = snapshot({'b':1, 'a':'hi'})
    assert [r.name for r in rows]==['a','b']
    assert rows[0].type=='str' and rows[0].value=="'hi'" and rows[0].path=='a'
    assert rows[1].type=='int' and rows[1].qualifier=='builtins'
def test_skips_hidden_and_dunder():
    rows = snapshot({'x':1, '_ih':2, '__y':3}, hidden=frozenset({'_ih'}))
    assert [r.name for r in rows]==['x']
def test_shape_populated():
    assert snapshot({'L':[1,2,3]})[0].shape=='3'
def test_container_flag_and_accessor():
    rows = snapshot({'d':{'a':1}, 'n':5})
    d, n = rows[0], rows[1]
    assert d.is_container is True and d.accessor==('d',)
    assert n.is_container is False and n.accessor==('n',)
def test_expand_dict_then_list():
    ns = {'d': {'x': 1, 'y': [10,20]}}
    kids = expand(ns, ('d',))
    assert [k.name for k in kids]==["'x'", "'y'"]
    assert kids[1].is_container and kids[1].accessor==('d',1) and kids[1].path=="d['y']"
    gk = expand(ns, ('d',1))
    assert [g.name for g in gk]==['0','1'] and gk[0].value=='10' and gk[0].path=="d['y'][0]"
def test_expand_truncates():
    kids = expand({'L': list(range(250))}, ('L',))
    assert len(kids)==101 and kids[-1].is_error is False and 'more' in kids[-1].value
def test_expand_sentinel_carries_offset():
    kids = expand({'L': list(range(250))}, ('L',))
    assert kids[-1].more_offset==100 and kids[-1].accessor==('L',)   # sentinel points at the next page
    assert all(k.more_offset is None for k in kids[:-1])
def test_expand_paging():
    ns = {'L': list(range(250))}
    p2 = expand(ns, ('L',), offset=100)
    assert p2[0].name=='100' and p2[0].value=='100' and p2[0].accessor==('L',100)   # true positional index
    assert len(p2)==101 and p2[-1].more_offset==200 and '50 more' in p2[-1].value
def test_expand_last_page_no_sentinel():
    p3 = expand({'L': list(range(250))}, ('L',), offset=200)
    assert len(p3)==50 and all(v.more_offset is None for v in p3)   # tail page, no load-more
def test_expand_non_container_empty():
    assert expand({'n':5}, ('n',))==[]
def test_gridables_are_grid_and_container():
    rows = snapshot({'arr': np.arange(4), 'df': pd.DataFrame({'a':[1]}), 'lst':[1,2]})
    by = {r.name:r for r in rows}
    assert by['arr'].is_grid and by['arr'].is_container   # both: grid link + tree
    assert by['df'].is_grid and by['df'].is_container
    assert by['lst'].is_container and not by['lst'].is_grid
def test_expand_ndarray_detail():
    kids = expand({'arr': np.arange(6).reshape(2,3)}, ('arr',))
    names = [k.name for k in kids]
    assert names[:3]==['shape','dtype','size'] and 'mean' in names and names[-2:]==['0','1']
    row0 = expand({'arr': np.arange(6).reshape(2,3)}, ('arr', names.index('0')))  # first row [0 1 2]
    elems = {k.name:k.value for k in row0}
    assert elems['0']=='0' and elems['2']=='2'   # numpy scalars render cleanly
def test_expand_dataframe_columns():
    kids = expand({'df': pd.DataFrame({'x':[1,2],'y':[3,4]})}, ('df',))
    names = [k.name for k in kids]
    assert 'shape' in names and names[-2:]==['x','y']
    col = expand({'df': pd.DataFrame({'x':[1,2],'y':[3,4]})}, ('df', names.index('x')))
    assert col[0].name=='dtype'   # column expands as a Series
def test_grid_page_walk():
    p = grid_page({'arr': np.arange(6).reshape(2,3)}, ('arr',))
    assert p['nrows']==2 and p['ncols']==3 and p['cells'][1]==['3','4','5']
def test_grid_page_non_gridable_none():
    assert grid_page({'n':5}, ('n',)) is None
def test_value_uses_provider():
    by = {r.name:r for r in snapshot({'p': pathlib.Path('/tmp/x'), 'n': 5})}
    assert by['p'].value==str(pathlib.Path('/tmp/x'))
    assert by['n'].value=='5'
def test_category():
    assert category('_x', 1)=='special' and category('__d__', 1)=='special' and category('In', 1)=='special'
    assert category('m', math)=='module'
    assert category('f', lambda x: x)=='function'
    assert category('C', int)=='type' and category('n', 5)=='data'   # classes group as 'type', values stay data
def test_profile_standard_groups():
    ns = {'n':5, 'f':(lambda x: x), 'm':math, '_h':9}
    view = profile_view(ns, profile='standard')
    labels = [l for l,_ in view]
    assert labels==[None,'Modules','Functions','Special Variables']
    assert [v.name for v in dict(view)[None]]==['n']   # only data at top level
def test_profile_minimal_hides_noise():
    ns = {'n':5, 'f':(lambda x: x), 'm':math, '_h':9}
    view = profile_view(ns, profile='minimal')
    assert [l for l,_ in view]==[None] and [v.name for v in dict(view)[None]]==['n']
def test_profile_full_promotes_funcs_and_modules():
    ns = {'n':5, 'f':(lambda x: x), 'm':math, '_h':9}
    labels = [l for l,_ in profile_view(ns, profile='full')]
    assert labels.count(None)==3 and labels[-1]=='Special Variables'   # data/modules/funcs flat
def test_profile_types_grouping():
    ns = {'n':5, 'C':int, 'D':dict}
    assert [l for l,_ in profile_view(ns, profile='minimal')]==[None]           # minimal hides types
    st = dict(profile_view(ns, profile='standard'))
    assert 'Types' in st and [v.name for v in st['Types']]==['C','D']           # standard groups them
    assert [v.name for v in st[None]]==['n']                                    # values stay flat at top
    assert [l for l,_ in profile_view(ns, profile='full')].count(None)==2       # full promotes types alongside data
def test_snapshot_filter_name():
    rows = snapshot({'alpha':1, 'beta':2, 'gamma':3}, name='a')
    assert [r.name for r in rows] == ['alpha', 'beta', 'gamma']   # 'a' substring, case-insensitive
    rows = snapshot({'alpha':1, 'beta':2}, name='ALPH')
    assert [r.name for r in rows] == ['alpha']
def test_snapshot_filter_type():
    rows = snapshot({'a':1, 'b':'x', 'c':2}, typ='int')
    assert [r.name for r in rows] == ['a', 'c']
def test_snapshot_filter_both():
    rows = snapshot({'a1':1, 'a2':'x', 'b1':3}, name='a', typ='int')
    assert [r.name for r in rows] == ['a1']
def test_snapshot_filter_no_match():
    assert snapshot({'a':1}, name='zzz') == []
for t in [test_sorted_and_fields,test_skips_hidden_and_dunder,test_shape_populated,
          test_container_flag_and_accessor,test_expand_dict_then_list,
          test_expand_truncates,test_expand_sentinel_carries_offset,test_expand_paging,
          test_expand_last_page_no_sentinel,test_expand_non_container_empty,
          test_gridables_are_grid_and_container,test_expand_ndarray_detail,
          test_expand_dataframe_columns,test_grid_page_walk,test_grid_page_non_gridable_none,
          test_value_uses_provider,test_category,test_profile_standard_groups,
          test_profile_minimal_hides_noise,test_profile_full_promotes_funcs_and_modules,
          test_profile_types_grouping,test_snapshot_filter_name,test_snapshot_filter_type,
          test_snapshot_filter_both,test_snapshot_filter_no_match]: t()

In [ ]:
#| export
from paar.core import VarInfo
from paar.reprs import get_shape, get_dtype
from paar.resolvers import resolve_for
from paar.grid import is_gridable, array_page
from paar.providers import value_str

MAX_CHILDREN = 100   # page size for one expand call

def _var_info(name, v, accessor, path):
    "Build a VarInfo for one value at the given accessor/path."
    try:
        return VarInfo(name=name, type=type(v).__name__,
                       qualifier=getattr(type(v), '__module__', '') or '',
                       value=value_str(v), shape=get_shape(v), dtype=get_dtype(v),
                       is_container=resolve_for(v) is not None,   # gridables are also expandable
                       is_grid=is_gridable(v), path=path, accessor=accessor)
    except Exception as e:
        return VarInfo(name=name, type='?', value=f'<error {e}>', is_error=True,
                       path=path, accessor=accessor)

def snapshot(ns, hidden=frozenset(), name=None, typ=None):
    "namespace dict -> sorted list[VarInfo], skipping hidden/dunder; optional name-substring & type-name filters."
    keys = sorted(k for k in ns if k not in hidden and not k.startswith('__'))
    if name: keys = [k for k in keys if name.lower() in k.lower()]
    if typ:  keys = [k for k in keys if ns[k].__class__.__name__.lower() == typ.lower()]
    return [_var_info(k, ns[k], (k,), k) for k in keys]

def _walk(ns, accessor):
    "Resolve a positional accessor (name, *idxs) to (object, readable_path). Raises KeyError/IndexError on bad path."
    name, *idxs = accessor
    obj, path = ns[name], name
    for i in idxs:
        r = resolve_for(obj)
        if r is None: raise KeyError(accessor)
        _, step, obj = r.children(obj)[i]
        path += step
    return obj, path

def expand(ns, accessor, offset=0, limit=MAX_CHILDREN):
    "One level of children of the value at accessor as VarInfo, windowed to kids[offset:offset+limit]; a load-more sentinel (more_offset set) trails a partial window."
    try:
        obj, path = _walk(ns, tuple(accessor))
    except Exception:
        return []
    r = resolve_for(obj)
    if r is None: return []
    kids = r.children(obj)
    out = [_var_info(nm, child, tuple(accessor)+(offset+i,), path+step)
           for i,(nm, step, child) in enumerate(kids[offset:offset+limit])]
    if len(kids) > offset+limit:
        out.append(VarInfo(name='…', type='', value=f'{len(kids)-(offset+limit)} more',
                           path=path, accessor=tuple(accessor), more_offset=offset+limit))
    return out

def grid_page(ns, accessor, roff=0, coff=0, rows=50, cols=50):
    "Walk accessor to a gridable object and return a page dict, or None."
    try:
        obj, _ = _walk(ns, tuple(accessor))
        if not is_gridable(obj): return None
        return array_page(obj, roff, coff, rows, cols)
    except Exception:
        return None

In [ ]:
#| export
import inspect

def category(name, v, hidden=frozenset()):
    "Classify a namespace entry: 'special' | 'module' | 'type' | 'function' | 'data'."
    if name in hidden or name.startswith('_') or name in ('In','Out','exit','quit','get_ipython'):
        return 'special'
    if inspect.ismodule(v): return 'module'
    if isinstance(v, type): return 'type'
    if callable(v): return 'function'
    return 'data'

GROUPS = {'special':'Special Variables', 'type':'Types', 'function':'Functions', 'module':'Modules'}

# profile -> per-category disposition: 'top' | 'group' | 'hidden' ('data' is always top-level)
PROFILES = {
    'minimal':  {'type':'hidden', 'function':'hidden', 'module':'hidden', 'special':'hidden'},
    'standard': {'type':'group',  'function':'group',  'module':'group',  'special':'group'},
    'full':     {'type':'top',    'function':'top',    'module':'top',    'special':'group'},
}

def profile_view(ns, hidden=frozenset(), profile='standard'):
    "Group a namespace per profile -> ordered list[(label|None, [VarInfo])] (label None renders flat)."
    disp = PROFILES.get(profile, PROFILES['standard'])
    cats = {}
    for k in sorted(ns):
        cats.setdefault(category(k, ns[k], hidden), []).append(_var_info(k, ns[k], (k,), k))
    out = [(None, cats.get('data', []))]
    for c in ('module', 'type', 'function', 'special'):
        vs = cats.get(c)
        if not vs: continue
        d = disp.get(c, 'group')
        if d == 'top':     out.append((None, vs))
        elif d == 'group': out.append((GROUPS[c], vs))
        # 'hidden' -> omit
    return [(lbl, vs) for lbl, vs in out if vs]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()